# Demo: Ingestion de Documentos

Este notebook demuestra cómo procesar documentos y construir un grafo de conocimiento.

In [ ]:
import sys
sys.path.append('..')

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor

## 1. Inicializar Neo4j

In [ ]:
neo4j = Neo4jManager()
neo4j.create_constraints()
neo4j.create_vector_index()

## 2. Cargar documento de ejemplo

In [ ]:
FILE_PATH = "../data/albatross.md"
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    markdown_text = f.read()

print(markdown_text)

## 3. Procesar documento

In [ ]:
processor = TextProcessor(neo4j, chunk_size=200, chunk_overlap=20)

processor.process_document(
    text=markdown_text,
    document_id="albatross",
    metadata={"title": "Albatross", "source": "Wikipedia"}
)

## 4. Verificar el grafo

In [ ]:
# Contar nodos
stats = neo4j.execute_query("""
MATCH (n)
RETURN labels(n)[0] as label, count(*) as count
ORDER BY count DESC
""")

for stat in stats:
    print(f"{stat['label']}: {stat['count']}")

In [ ]:
# Ver algunas entidades
entities = neo4j.execute_query("""
MATCH (e:Entity)
RETURN e.name as name, e.type as type, e.summary as summary
LIMIT 10
""")

for entity in entities:
    print(f"\n{entity['name']} ({entity['type']})")
    print(f"  {entity['summary'][:100]}..." if entity['summary'] else "  No summary")

In [ ]:
# Ver algunas relaciones
relationships = neo4j.execute_query("""
MATCH (s:Entity)-[r:RELATIONSHIP]->(t:Entity)
RETURN s.name as source, r.type as type, t.name as target, r.summary as summary
LIMIT 10
""")

for rel in relationships:
    print(f"\n{rel['source']} -[{rel['type']}]-> {rel['target']}")
    print(f"  {rel['summary'][:100]}..." if rel['summary'] else "  No summary")

## 5. Cleanup (opcional)

In [ ]:
# Descomentar para limpiar la base de datos
# neo4j.execute_query("MATCH (n) DETACH DELETE n")
neo4j.close()